# MedDecide — Bayesian Network Exploration

This notebook walks through:
1. Training the BN on the Mendeley dataset
2. Visualizing the CPTs
3. Running inference experiments
4. Analyzing uncertainty metrics

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
print('Setup complete')

## 1. Train the Bayesian Network

In [ ]:
from src.reasoner.bn_trainer import BayesianNetworkTrainer

trainer = BayesianNetworkTrainer(laplace_alpha=1.0)

# Create demo dataset (replace with your Mendeley CSV path)
demo_path = trainer.create_demo_dataset(Path('../data/demo_dataset.csv'))
model = trainer.train(demo_path, Path('../models/bn_model.pkl'))

print(f'Trained model with {len(model.nodes())} nodes and {len(model.edges())} edges')

## 2. Inspect the CPTs

In [ ]:
# Print disease prior
disease_cpd = model.get_cpds('disease')
states = disease_cpd.state_names['disease']
priors = disease_cpd.values.flatten()

prior_df = pd.DataFrame({'Disease': states, 'Prior': priors}).sort_values('Prior', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(prior_df['Disease'][:15], prior_df['Prior'][:15], color='#6366f1')
ax.set_xlabel('Prior Probability')
ax.set_title('Disease Prior Distribution (Top 15)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize P(fever | disease) for each disease
fever_cpd = model.get_cpds('fever')
disease_states = fever_cpd.state_names['disease']

# P(fever=1 | disease)
p_fever_given_disease = fever_cpd.values[1]  # index 1 = fever present

fever_df = pd.DataFrame({
    'Disease': disease_states,
    'P(fever=1|disease)': p_fever_given_disease
}).sort_values('P(fever=1|disease)', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(fever_df['Disease'][:15], fever_df['P(fever=1|disease)'][:15])
for bar, val in zip(bars, fever_df['P(fever=1|disease)'][:15]):
    bar.set_color(plt.cm.RdYlGn(val))
ax.set_xlabel('P(fever=1 | disease)')
ax.set_title('Probability of Fever Given Disease')
plt.tight_layout()
plt.show()

## 3. Run Inference Experiments

In [ ]:
from src.reasoner.bn_inference import BayesianReasoner

reasoner = BayesianReasoner(Path('../models/bn_model.pkl'))

# Test cases
test_cases = [
    {'name': 'Flu-like', 'evidence': {'fever': 1, 'body_ache': 1, 'chills': 1, 'headache': 1}},
    {'name': 'GI Bug', 'evidence': {'nausea': 1, 'vomiting': 1, 'diarrhea': 1, 'abdominal_pain': 1}},
    {'name': 'Chest', 'evidence': {'chest_pain': 1, 'shortness_of_breath': 1, 'sweating': 1}},
    {'name': 'Meningitis', 'evidence': {'severe_headache': 1, 'neck_stiffness': 1, 'fever': 1}},
]

results = {}
for case in test_cases:
    result = reasoner.infer(hard_evidence=case['evidence'], top_k=5)
    results[case['name']] = result
    print(f"\n{case['name']} ({case['evidence']}):")
    for disease, prob in result.top_diseases[:3]:
        print(f'  {disease}: {prob:.3f}')

In [ ]:
# Heatmap: top diseases for each test case
all_diseases = set()
for r in results.values():
    all_diseases.update(d for d, _ in r.top_diseases[:5])
all_diseases = sorted(all_diseases)

matrix = []
case_names = list(results.keys())
for name in case_names:
    probs_dict = dict(results[name].top_diseases)
    row = [probs_dict.get(d, 0) for d in all_diseases]
    matrix.append(row)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    matrix, xticklabels=all_diseases, yticklabels=case_names,
    annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
    cbar_kws={'label': 'P(disease)'}
)
ax.set_title('Disease Probabilities by Test Case')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Uncertainty Analysis

In [ ]:
from src.reasoner.uncertainty import UncertaintyDetector

detector = UncertaintyDetector()

print(f"{'Case':20s} {'Entropy':10s} {'MaxProb':10s} {'Confidence':12s} {'Uncertain':10s} {'Novel'}")
print('-' * 70)

for name, result in results.items():
    report = detector.analyze(result)
    print(f"{name:20s} {report.entropy:10.3f} {report.max_prob:10.3f} "
          f"{report.confidence_label:12s} {str(report.is_uncertain):10s} {report.is_novel}")

## 5. Full Pipeline Test

In [ ]:
from src.pipeline import MedDecidePipeline

pipeline = MedDecidePipeline(use_llm=False)

text = 'I have had a high fever, severe headache, chills, and body aches for 2 days. Also feeling very tired.'
result = pipeline.run(text)

print(f'Input: {text}')
print(f'\nTop diseases:')
for d, p in result.inference_result.top_diseases[:5]:
    print(f'  {d}: {p:.3f}')

print(f'\nRecommended action: {result.recommended_label}')
print(f'Confidence: {result.confidence_label}')
print(f'\nSummary: {result.summary}')